# Standalone Inference Notebook — PM2.5 Forecasting

**Purpose:** Load a pre-trained `best_model.pt` (from a previous Kaggle run) and generate a correct `preds.npy` submission file.

**How to use on Kaggle:**
1. Add **three** datasets to this notebook:
   - Competition data (`aisehack-theme-2`) — contains `test_in/`, `raw/`, `stats/`
   - Source dataset (`murtuza-pm25-src30`) — contains `Murtuza/src/`, `Murtuza/configs/`
   - A dataset containing your saved `best_model.pt` — upload it separately
2. Update `BEST_MODEL_PATH` in Cell 1 to point to where `best_model.pt` is mounted
3. Run all cells — `preds.npy` will appear in `/kaggle/working/`

**Preprocessing fixes applied (vs original `pino_baseline.ipynb`):**
- **cpm25 future masking**: hours 10–16 set to **zeros** (not persistence, not oracle values) — matches `SlidingWindowTensorDataset` training masking exactly  
- **Lag boundary fix**: `lag_1/3/6` boundary steps zero-padded instead of wrapping with `np.roll`  
- **PBLH normalization**: always uses global official bounds (no local per-month scaling at test time)  
- **TFNO2D output clamped** to `[0, 1]` normalized space before denormalization


In [ ]:
# ── Cell 1: Bootstrap — paths, imports ─────────────────────────────────────
import os, sys
import numpy as np
import torch

# ── Kaggle paths — adjust MURTUZA_DIR, BEST_MODEL_PATH, PIXEL_STATS_PATH ───
KAGGLE_DATA_ROOT  = "/kaggle/input/datasets/khushisingh942004/aisehack"
MURTUZA_DIR       = "/kaggle/input/datasets/sasidhar412/murtuza-pm25-src30/ANRF_AISEHack_Code/Murtuza"

# ⚠️  Update these to the mount paths of your uploaded checkpoint dataset:
BEST_MODEL_PATH   = "/kaggle/input/murtuza-best-model/best_model.pt"
# pixel_stats.npz must be uploaded alongside best_model.pt (same dataset folder).
PIXEL_STATS_PATH  = "/kaggle/input/murtuza-best-model/pixel_stats.npz"

OUT_DIR           = "/kaggle/working"
PRED_PATH         = os.path.join(OUT_DIR, "preds.npy")

# ── Local fallback (non-Kaggle debugging) ────────────────────────────────────
if not os.path.exists("/kaggle"):
    MURTUZA_DIR      = os.path.abspath("../Murtuza")
    KAGGLE_DATA_ROOT = os.path.abspath("../../aisehack-theme-2")
    BEST_MODEL_PATH  = os.path.abspath("../outputs/models/best_model.pt")
    PIXEL_STATS_PATH = os.path.abspath("../outputs/models/pixel_stats.npz")
    OUT_DIR          = os.path.abspath("../outputs/submissions")
    PRED_PATH        = os.path.join(OUT_DIR, "preds.npy")

if MURTUZA_DIR not in sys.path:
    sys.path.insert(0, MURTUZA_DIR)

os.makedirs(OUT_DIR, exist_ok=True)

assert os.path.exists(MURTUZA_DIR),      f"Source dir not found: {MURTUZA_DIR}"
assert os.path.exists(KAGGLE_DATA_ROOT), f"Data root not found: {KAGGLE_DATA_ROOT}"
assert os.path.exists(BEST_MODEL_PATH),  (
    f"best_model.pt not found: {BEST_MODEL_PATH}\n"
    "→ Upload your best_model.pt as a Kaggle dataset and update BEST_MODEL_PATH above."
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device          : {DEVICE}")
print(f"Data            : {KAGGLE_DATA_ROOT}")
print(f"Source          : {MURTUZA_DIR}")
print(f"Model           : {BEST_MODEL_PATH}")
print(f"Pixel stats     : {PIXEL_STATS_PATH} ({'found' if os.path.exists(PIXEL_STATS_PATH) else 'NOT FOUND — will use global min-max'})")
print(f"Output          : {PRED_PATH}")


In [ ]:
# ── Cell 2: Load config, official bounds, and pixel stats ──────────────────
from src.config import load_config
from src.data import load_minmax_bounds, FALLBACK_OFFICIAL_BOUNDS, load_pixel_stats

cfg = load_config()
cfg['paths']['data'] = KAGGLE_DATA_ROOT
cfg['device'] = DEVICE

bounds = load_minmax_bounds(cfg)
fmin_cpm = bounds['cpm25'][0]
fmax_cpm = bounds['cpm25'][1]

# Load per-pixel z-score stats (Topic 2) if available
USE_PIXEL_NORM  = cfg.get('data', {}).get('use_pixel_norm', False)
RESIDUAL_MODE   = cfg['training'].get('residual_target', False)
T_IN_CPM        = cfg['time']['t_in_cpm']

pixel_stats_inf = None
if USE_PIXEL_NORM and os.path.exists(PIXEL_STATS_PATH):
    pixel_stats_inf = load_pixel_stats(PIXEL_STATS_PATH)
    print(f"Per-pixel stats loaded: {PIXEL_STATS_PATH}")
elif USE_PIXEL_NORM:
    print("Warning: pixel_stats.npz not found. Falling back to global min-max denorm.")
    USE_PIXEL_NORM = False

print(f"cpm25 bounds       : [{fmin_cpm}, {fmax_cpm}]")
print(f"Pixel normalization: {'ENABLED (Topic 2)' if USE_PIXEL_NORM else 'OFF (global min-max)'}")
print(f"Residual target    : {'ENABLED (Topic 8)' if RESIDUAL_MODE else 'OFF'}")
print(f"Config months      : {cfg['data']['months']}")


In [ ]:
# ── Cell 3: Inspect checkpoint, infer channel layout, build model ────────────
from src.model import build_model

state = torch.load(BEST_MODEL_PATH, map_location=DEVICE)

# Infer expected flat input channels directly from checkpoint.
if 'lift.weight' in state:
    expected_flat = int(state['lift.weight'].shape[1])
else:
    expected_flat = int(cfg.get('tensor_channels', 27 * cfg['time'].get('t_in', 16)))

T_MODEL = int(cfg['time'].get('t_in', 16))
if expected_flat % T_MODEL != 0:
    raise RuntimeError(f"Checkpoint input channels {expected_flat} not divisible by T_MODEL={T_MODEL}")

N_CHANNELS = expected_flat // T_MODEL

# Channel layout versions:
#   v3 (current): 15 base + 12 engineered (ventilation_index added) = 27
#   v2 (prev):    15 base + 11 engineered (no ventilation_index)    = 26
#   v1 (legacy):  16 base + 12 engineered                           = 28
CURRENT_BASE_FEATS = list(cfg['features']['base'])
LEGACY_BASE_FEATS  = [
    'cpm25', 'u10', 'v10', 'pblh', 'rain', 't2', 'q2', 'swdown', 'psfc',
    'PM25', 'SO2', 'NOx', 'NH3', 'NMVOC_e', 'NMVOC_finn', 'bio',
]

if N_CHANNELS == 27:
    BASE_FEATS     = CURRENT_BASE_FEATS
    ENGINEER_MODE  = 'current27'    # 15 base + 12 engineered (incl. ventilation_index)
elif N_CHANNELS == 26:
    BASE_FEATS     = CURRENT_BASE_FEATS
    ENGINEER_MODE  = 'prev26'       # 15 base + 11 engineered (no ventilation_index)
elif N_CHANNELS == 28:
    BASE_FEATS     = LEGACY_BASE_FEATS
    ENGINEER_MODE  = 'legacy28'
    for feat in BASE_FEATS:
        if feat not in bounds:
            bounds[feat] = FALLBACK_OFFICIAL_BOUNDS[feat]
else:
    raise RuntimeError(
        f"Unsupported checkpoint channel count: {N_CHANNELS}. "
        "Supported: 27 (current), 26 (prev), 28 (legacy)."
    )

cfg['tensor_channels'] = expected_flat
model = build_model(cfg).to(DEVICE)
model.load_state_dict(state)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"Checkpoint flat in-channels : {expected_flat}")
print(f"T_MODEL                     : {T_MODEL}")
print(f"N_CHANNELS                  : {N_CHANNELS}")
print(f"Preprocessing mode          : {ENGINEER_MODE}")
print(f"Base feature count          : {len(BASE_FEATS)}")
print(f"Model type                  : {cfg['model']['type']}")
print(f"Parameters                  : {total_params:,}")
print(f"Weights loaded              : {BEST_MODEL_PATH}")


In [ ]:
# ── Cell 4: Memory-safe chunked preprocessing helpers ───────────────────────
import gc
from src.data import (
    _transform_bounds, preprocess_feature, preprocess_feature_pixelwise,
    get_hotspot_maps,
)

TEST_DIR   = os.path.join(KAGGLE_DATA_ROOT, 'test_in')
N_TEST     = cfg['data']['test_samples']
T_IN_MET   = cfg['time']['t_in_met']
H, W       = 140, 124

# Conservative batch size for Kaggle notebook RAM safety.
BATCH_SIZE = min(int(cfg['training'].get('batch_size_test', 16)), 4)
print(f"Using memory-safe inference batch size: {BATCH_SIZE}")

HOTSPOT_PRIOR = None
LAT_MAP       = None
LON_MAP       = None

if ENGINEER_MODE in ('current27', 'prev26'):
    HOTSPOT_PRIOR, _ = get_hotspot_maps(cfg)
    print(f"Prepared hotspot prior for {ENGINEER_MODE} checkpoint")
elif ENGINEER_MODE == 'legacy28':
    ll      = np.load(os.path.join(KAGGLE_DATA_ROOT, 'raw', 'lat_long.npy')).astype(np.float32)
    LAT_MAP = ll[:, :, 0]
    LON_MAP = ll[:, :, 1]
    LAT_MAP = (LAT_MAP - LAT_MAP.min()) / (LAT_MAP.max() - LAT_MAP.min() + np.float32(1e-8))
    LON_MAP = (LON_MAP - LON_MAP.min()) / (LON_MAP.max() - LON_MAP.min() + np.float32(1e-8))
    print("Prepared normalized lat/lon maps for legacy 28-channel checkpoint")


def _normalize_feat(feat, arr, bs):
    """Applies pixel normalization or global min-max depending on config."""
    if USE_PIXEL_NORM and pixel_stats_inf is not None and feat != 'cpm25':
        return preprocess_feature_pixelwise(arr, feat, pixel_stats_inf)
    return preprocess_feature(arr, feat, bounds, month=None)


def build_test_chunk(start: int, end: int) -> np.ndarray:
    """Build one inference-ready chunk: (B, N_CHANNELS, T_MODEL, H, W)."""
    bs           = end - start
    base_chunks  = []

    for feat in BASE_FEATS:
        if feat == 'rain_mask':
            rain_raw = np.load(os.path.join(TEST_DIR, 'rain.npy'), mmap_mode='r')[start:end]
            base_chunks.append((np.asarray(rain_raw, dtype=np.float32) > 0).astype(np.float32))
            continue

        raw = np.asarray(np.load(os.path.join(TEST_DIR, f'{feat}.npy'), mmap_mode='r')[start:end],
                         dtype=np.float32)

        if feat == 'cpm25':
            if USE_PIXEL_NORM and pixel_stats_inf is not None:
                cpm_norm = preprocess_feature_pixelwise(raw, feat, pixel_stats_inf)
            else:
                cpm_log  = np.log1p(np.clip(raw, 0.0, None))
                lo_t, hi_t = _transform_bounds(fmin_cpm, fmax_cpm, 'cpm25')
                cpm_norm = np.clip((cpm_log - lo_t) / (hi_t - lo_t + 1e-8), 0.0, 1.0).astype(np.float32)
            # Fill future cpm25 hours with ZEROS — matches training masking exactly
            zeros = np.zeros((bs, T_IN_MET - T_IN_CPM, H, W), dtype=np.float32)
            base_chunks.append(np.concatenate([cpm_norm, zeros], axis=1))
            continue

        base_chunks.append(_normalize_feat(feat, raw, bs))

    x = np.stack(base_chunks, axis=1).astype(np.float32)    # (B, C_base, T_IN_MET, H, W)
    x = x[:, :, :T_MODEL, :, :]                             # (B, C_base, T_MODEL, H, W)

    feat_idx   = {name: i for i, name in enumerate(BASE_FEATS)}
    u10        = x[:, feat_idx['u10']]
    v10        = x[:, feat_idx['v10']]
    t2         = x[:, feat_idx['t2']]
    q2         = x[:, feat_idx['q2']]
    pblh       = x[:, feat_idx['pblh']]
    cpm25      = x[:, feat_idx['cpm25']]
    wind_speed = np.sqrt(u10 ** 2 + v10 ** 2)
    wind_dir   = np.arctan2(v10, u10)

    # Topic 6: Ventilation Index = PBLH × wind_speed
    ventilation_index = pblh * wind_speed

    denom_safe = np.where(
        np.abs(t2 - np.float32(29.65)) < np.float32(1e-3),
        np.sign(t2 - np.float32(29.65) + np.float32(1e-3)) * np.float32(1e-3),
        t2 - np.float32(29.65),
    ).astype(np.float32)
    exponent = np.clip(
        np.float32(17.67) * (t2 - np.float32(273.15)) / denom_safe,
        np.float32(-87.0), np.float32(87.0),
    )
    with np.errstate(over='ignore', invalid='ignore'):
        rh = q2 / (np.float32(0.622) + np.float32(0.378) * q2 + np.float32(1e-8)) * np.exp(exponent)
    np.nan_to_num(rh, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    rh = np.clip(rh, np.float32(0.0), np.float32(1.5))

    hour      = (np.arange(T_MODEL, dtype=np.float32) % 24)
    hour_sin  = np.broadcast_to(np.sin(np.float32(2*np.pi) * hour / 24.0).reshape(1,T_MODEL,1,1), (bs,T_MODEL,H,W)).copy()
    hour_cos  = np.broadcast_to(np.cos(np.float32(2*np.pi) * hour / 24.0).reshape(1,T_MODEL,1,1), (bs,T_MODEL,H,W)).copy()
    month_sin = np.full((bs, T_MODEL, H, W), np.sin(np.float32(2*np.pi/12.0)), dtype=np.float32)
    month_cos = np.full((bs, T_MODEL, H, W), np.cos(np.float32(2*np.pi/12.0)), dtype=np.float32)

    # Lag features (boundary-zero-padded, not wrapped)
    lag_1 = np.roll(cpm25, 1, axis=1); lag_1[:, :1] = 0.0
    lag_3 = np.roll(cpm25, 3, axis=1); lag_3[:, :3] = 0.0
    lag_6 = np.roll(cpm25, 6, axis=1); lag_6[:, :6] = 0.0

    if ENGINEER_MODE == 'current27':
        hotspot = np.broadcast_to(HOTSPOT_PRIOR.reshape(1,1,H,W), (bs,T_MODEL,H,W)).copy().astype(np.float32)
        engineered = np.stack([
            wind_speed, wind_dir, ventilation_index,
            rh, hour_sin, hour_cos, month_sin, month_cos,
            hotspot, lag_1, lag_3, lag_6,               # 12 engineered channels
        ], axis=1).astype(np.float32)
    elif ENGINEER_MODE == 'prev26':
        # Legacy 26-channel checkpoint without ventilation_index
        hotspot = np.broadcast_to(HOTSPOT_PRIOR.reshape(1,1,H,W), (bs,T_MODEL,H,W)).copy().astype(np.float32)
        engineered = np.stack([
            wind_speed, wind_dir,
            rh, hour_sin, hour_cos, month_sin, month_cos,
            hotspot, lag_1, lag_3, lag_6,               # 11 engineered channels
        ], axis=1).astype(np.float32)
    elif ENGINEER_MODE == 'legacy28':
        lat = np.broadcast_to(LAT_MAP.reshape(1,1,H,W), (bs,T_MODEL,H,W)).copy().astype(np.float32)
        lon = np.broadcast_to(LON_MAP.reshape(1,1,H,W), (bs,T_MODEL,H,W)).copy().astype(np.float32)
        engineered = np.stack([
            wind_speed, wind_dir, rh,
            hour_sin, hour_cos, month_sin, month_cos,
            lat, lon, lag_1, lag_3, lag_6,
        ], axis=1).astype(np.float32)
    else:
        raise RuntimeError(f"Unsupported engineer mode: {ENGINEER_MODE}")

    out = np.concatenate([x, engineered], axis=1).astype(np.float32)
    np.nan_to_num(out, copy=False, nan=0.0, posinf=0.0, neginf=0.0)

    expected = (bs, N_CHANNELS, T_MODEL, H, W)
    assert out.shape == expected, f"Expected {expected}, got {out.shape}"
    return out


In [ ]:
# ── Cell 5: Smoke test one chunk before full inference ──────────────────────
smoke_end = min(BATCH_SIZE, N_TEST)
x_smoke = build_test_chunk(0, smoke_end)
print(f"Smoke-test chunk shape : {x_smoke.shape}")
print(f"Smoke-test value range : [{x_smoke.min():.3f}, {x_smoke.max():.3f}]")
print(f"Smoke-test NaN/Inf     : {int(np.sum(~np.isfinite(x_smoke)))}")

del x_smoke
gc.collect()
print("Chunk builder sanity check passed.")

In [ ]:
# ── Cell 6: Streaming inference to disk-backed .npy memmap ──────────────────
from src.data import denormalize, denormalize_pixelwise

TMP_PRED_PATH = os.path.join(OUT_DIR, 'preds_tmp.npy')
if os.path.exists(TMP_PRED_PATH):
    os.remove(TMP_PRED_PATH)
if os.path.exists(PRED_PATH):
    os.remove(PRED_PATH)

# Disk-backed output avoids holding ~1.1 GB preds array in RAM.
preds_mm = np.lib.format.open_memmap(
    TMP_PRED_PATH,
    mode  = 'w+',
    dtype = np.float32,
    shape = (N_TEST, H, W, T_MODEL),
)

model.eval()
with torch.no_grad():
    for i in range(0, N_TEST, BATCH_SIZE):
        j       = min(i + BATCH_SIZE, N_TEST)
        x_batch = build_test_chunk(i, j)
        batch   = torch.from_numpy(x_batch).to(DEVICE)

        pred_out = model(batch)          # (B, H, W, T_out) — delta or absolute (z-score or [0,1])

        # ── Topic 8: residual reconstruction ─────────────────────────────────
        if RESIDUAL_MODE:
            last_obs_z = batch[:, 0, T_IN_CPM - 1, :, :]          # (B, H, W) z-score
            pred_abs   = (last_obs_z.unsqueeze(-1) + pred_out).cpu().numpy()   # (B, H, W, T)
        else:
            pred_abs = pred_out.cpu().numpy()                       # (B, H, W, T)

        # ── Denormalization ───────────────────────────────────────────────────
        if USE_PIXEL_NORM and pixel_stats_inf is not None:
            pred_phys = denormalize_pixelwise(pred_abs, 'cpm25', pixel_stats_inf)
        else:
            pred_phys = denormalize(pred_abs, fmin_cpm, fmax_cpm, feat='cpm25')

        pred_phys = np.clip(pred_phys, 0.0, None).astype(np.float32)

        preds_mm[i:j] = pred_phys
        preds_mm.flush()

        if i == 0:
            print(f"[Batch 0] input {batch.shape} → raw output {pred_out.shape}")
            print(f"          denorm mode: {'pixel-z-score' if USE_PIXEL_NORM else 'global min-max'}")
            print(f"          residual   : {RESIDUAL_MODE}")
            print(f"          pred range : [{pred_phys.min():.1f}, {pred_phys.max():.1f}] µg/m³")
        print(f"Processed {j}/{N_TEST} samples", end='\r')

        del x_batch, batch, pred_out, pred_abs, pred_phys
        gc.collect()
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

print("\nInference complete.")
del preds_mm
os.replace(TMP_PRED_PATH, PRED_PATH)
print(f"Streaming predictions written to: {PRED_PATH}")


In [ ]:
# ── Cell 7: Verify saved preds.npy without loading all of it into RAM ───────
loaded = np.load(PRED_PATH, mmap_mode='r')
assert loaded.shape == (N_TEST, H, W, T_MODEL), f"Read-back shape mismatch: {loaded.shape}"

sample = np.asarray(loaded[: min(8, N_TEST)])
assert np.isfinite(sample).all(), "Sampled predictions contain NaN/Inf"

size_mb = os.path.getsize(PRED_PATH) / (1024**2)
print("═" * 55)
print(f"  preds.npy saved to : {PRED_PATH}")
print(f"  Shape              : {loaded.shape}")
print(f"  dtype              : {loaded.dtype}")
print(f"  File size          : {size_mb:.2f} MB")
print(f"  Sample range       : [{sample.min():.1f}, {sample.max():.1f}] µg/m³")
print(f"  Sample mean PM2.5  : {sample.mean():.2f} µg/m³")
print("═" * 55)
print("Ready to submit. Download preds.npy from /kaggle/working/ and upload it.")